<a href="https://colab.research.google.com/github/hsrkl/techrush26/blob/notebooks/FF_detection_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "credit_card_transactions-ibm_v2.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "ealtman2019/credit-card-transactions",
  file_path
)
non_fraud = df[df['Is Fraud?'] == 'No']
fraud = df[df['Is Fraud?'] == 'Yes']

non_fraud_reduced = non_fraud.sample(n=len(non_fraud)-23000000, random_state=42)

df = pd.concat([fraud, non_fraud_reduced]).sample(frac=1, random_state=42).reset_index(drop=True)

print(df.shape)
print(df['Is Fraud?'].value_counts())

#print(df.head(10))
#print(df.shape)
#print(df[df['Is Fraud?'] == 'No'].shape[0])

print(df.head())

Using Colab cache for faster access to the 'credit-card-transactions' dataset.
(1386900, 15)
Is Fraud?
No     1357143
Yes      29757
Name: count, dtype: int64
   User  Card  Year  Month  Day   Time  Amount            Use Chip  \
0  1560     1  2013      7   18  14:59  $20.24  Online Transaction   
1   488     0  2003      8   23  12:31  $15.43   Swipe Transaction   
2  1634     2  2019      8   30  19:09   $2.09    Chip Transaction   
3  1429     1  2018     11    2  14:29  $52.47  Online Transaction   
4  1276     0  2019      9    9  10:53  $37.68  Online Transaction   

         Merchant Name Merchant City Merchant State      Zip   MCC Errors?  \
0 -6458444334611773637        ONLINE            NaN      NaN  4784     NaN   
1  6621275923246397337   Los Angeles             CA  90033.0  5814     NaN   
2 -6571010470072147219      Meridian             MS  39301.0  5499     NaN   
3 -2088492411650162548        ONLINE            NaN      NaN  4784     NaN   
4  4241336128694185533        

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, f1_score

# 2. Clean + encode
if df['Amount'].dtype == object:
    df['Amount'] = df['Amount'].str.replace('$', '', regex=False).astype(float)

if df['Is Fraud?'].dtype == object:
    df['Is Fraud?'] = df['Is Fraud?'].map({'Yes': 1, 'No': 0})

if 'Time' in df.columns:
    df['Hour'] = df['Time'].str.split(':').str[0].astype(int)
    df.drop(columns=['Time'], inplace=True)

if 'Merchant Name' in df.columns:
    df['merchant_freq'] = df['Merchant Name'].map(df['Merchant Name'].value_counts())

if 'Merchant City' in df.columns:
    df['merchant_city_freq'] = df['Merchant City'].map(df['Merchant City'].value_counts())

if 'Merchant State' in df.columns:
    df['merchant_state_enc'] = LabelEncoder().fit_transform(df['Merchant State'].astype(str))

if 'Use Chip' in df.columns:
    df['use_chip_enc'] = LabelEncoder().fit_transform(df['Use Chip'].astype(str))

if 'Errors?' in df.columns:
    df['has_error'] = df['Errors?'].fillna('').ne('').astype(int)

# 3. Drop raw text columns
cols_to_drop = [c for c in ['Use Chip', 'Merchant Name', 'Merchant City', 'Merchant State', 'Errors?', 'Zip'] if c in df.columns]
df_model = df.drop(columns=cols_to_drop)
print("Model-ready shape:", df_model.shape)
print(df_model.dtypes)

# 4. Split features/target
X = df_model.drop(columns=['Is Fraud?'])
y = df_model['Is Fraud?']

# 5. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

# 6. scale_pos_weight
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", scale_pos_weight)

# 7. Train XGBoost
model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    random_state=42
)
model.fit(X_train, y_train)

# 8. Evaluate
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, digits=4))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("PR-AUC:", average_precision_score(y_test, y_proba))
print(df['Merchant State'].value_counts().idxmax())

Model-ready shape: (1386900, 14)
User                    int64
Card                    int64
Year                    int64
Month                   int64
Day                     int64
Amount                float64
MCC                     int64
Is Fraud?               int64
Hour                    int64
merchant_freq           int64
merchant_city_freq      int64
merchant_state_enc      int64
use_chip_enc            int64
has_error               int64
dtype: object
Train: (1109520, 13) | Test: (277380, 13)
scale_pos_weight: 45.606737797193986

--- Classification Report ---
              precision    recall  f1-score   support

           0     0.9991    0.9863    0.9927    271429
           1     0.6054    0.9614    0.7429      5951

    accuracy                         0.9857    277380
   macro avg     0.8023    0.9738    0.8678    277380
weighted avg     0.9907    0.9857    0.9873    277380

F1: 0.7429387702097267
ROC-AUC: 0.9977072846166365
PR-AUC: 0.952280069784411
CA


In [ ]:
print(df['Merchant State'].unique())

[nan 'CA' 'MS' 'PA' 'FL' 'Italy' 'GA' 'AL' 'TN' 'SC' 'MO' 'VA' 'KS' 'NJ'
 'TX' 'OH' 'MI' 'WI' 'ME' 'CO' 'MA' 'AZ' 'NH' 'NM' 'NY' 'KY' 'IL' 'VT'
 'WA' 'NC' 'IN' 'SD' 'United Kingdom' 'UT' 'LA' 'CT' 'IA' 'OK' 'WV' 'MT'
 'NE' 'HI' 'NV' 'MD' 'MN' 'ND' 'RI' 'OR' 'AR' 'DE' 'ID' 'India' 'Mexico'
 'Malaysia' 'WY' 'Algeria' 'China' 'Georgia' 'Aruba' 'Haiti' 'DC' 'Japan'
 'Spain' 'Germany' 'Colombia' 'AK' 'United Arab Emirates' 'Canada'
 'Serbia' 'Australia' 'France' 'Oman' 'Turkey' 'Netherlands' 'Taiwan'
 'Dominican Republic' 'Brazil' 'Austria' 'Kenya' 'Philippines' 'Hong Kong'
 'Belgium' 'Thailand' 'Jamaica' 'South Korea' 'Switzerland' 'Israel'
 'Argentina' 'Greece' 'Nigeria' 'Finland' 'Costa Rica' 'Pakistan'
 'Guatemala' 'Hungary' 'South Africa' 'Ireland' 'Poland' 'Tuvalu' 'Fiji'
 'Macedonia' 'Egypt' 'Ukraine' 'New Zealand' 'Slovenia' 'Lithuania'
 'Latvia' 'Sweden' 'Singapore' 'Bangladesh' 'The Bahamas' 'Vatican City'
 'Czech Republic' 'Liberia' 'Dominica' 'Norway' 'Peru' 'Uganda' 'Portugal'


In [ ]:
import pandas as pd
import numpy as np

# Lat/Long for US States
us_state_coords = {
    'AL': (32.806671, -86.791130), 'AK': (61.370716, -152.404419),
    'AZ': (33.729759, -111.431221), 'AR': (34.969704, -92.373123),
    'CA': (36.116203, -119.681564), 'CO': (39.059811, -105.311104),
    'CT': (41.597782, -72.755371), 'DE': (39.318523, -75.507141),
    'DC': (38.897438, -77.026817), 'FL': (27.766279, -81.686783),
    'GA': (33.040619, -83.643074), 'HI': (21.094318, -157.498337),
    'ID': (44.240459, -114.478828), 'IL': (40.349457, -88.986137),
    'IN': (39.849426, -86.258278), 'IA': (42.011539, -93.210526),
    'KS': (38.526600, -96.726486), 'KY': (37.668140, -84.670067),
    'LA': (31.169960, -91.867805), 'ME': (44.693947, -69.381927),
    'MD': (39.063946, -76.802101), 'MA': (42.230171, -71.530106),
    'MI': (43.326618, -84.536095), 'MN': (45.694454, -93.900192),
    'MS': (32.741646, -89.678696), 'MO': (38.456085, -92.288368),
    'MT': (46.921925, -110.454353), 'NE': (41.125370, -98.268082),
    'NV': (38.313515, -117.055374), 'NH': (43.452492, -71.563896),
    'NJ': (40.298904, -74.521011), 'NM': (34.840515, -106.248482),
    'NY': (42.165726, -74.948051), 'NC': (35.630066, -79.806419),
    'ND': (47.528912, -99.784012), 'OH': (40.388783, -82.764915),
    'OK': (35.565342, -96.928917), 'OR': (44.572021, -122.070938),
    'PA': (40.590752, -77.209755), 'RI': (41.680893, -71.511780),
    'SC': (33.856892, -80.945007), 'SD': (44.299782, -99.438828),
    'TN': (35.747845, -86.692345), 'TX': (31.054487, -97.563461),
    'UT': (40.150032, -111.862434), 'VT': (44.045876, -72.710686),
    'VA': (37.769337, -78.169968), 'WA': (47.400902, -121.490494),
    'WV': (38.491226, -80.954453), 'WI': (44.268543, -89.616508),
    'WY': (42.755966, -107.302490)
}

# Lat/Long for Countries
country_coords = {
    'Italy': (41.8719, 12.5674), 'United Kingdom': (55.3781, -3.4360), 'Online':((36.116203, -119.681564)),
    'India': (20.5937, 78.9629), 'Mexico': (23.6345, -102.5528),
    'Malaysia': (4.2105, 101.9758), 'Algeria': (28.0339, 1.6596),
    'China': (35.8617, 104.1954), 'Georgia': (42.3154, 43.3569),
    'Aruba': (12.5211, -69.9683), 'Haiti': (18.9712, -72.2852),
    'Japan': (36.2048, 138.2529), 'Spain': (40.4637, -3.7492),
    'Germany': (51.1657, 10.4515), 'Colombia': (4.5709, -74.2973),
    'United Arab Emirates': (23.4241, 53.8478), 'Canada': (56.1304, -106.3468),
    'Serbia': (44.0165, 21.0059), 'Australia': (-25.2744, 133.7751),
    'France': (46.2276, 2.2137), 'Oman': (21.4735, 55.9754),
    'Turkey': (38.9637, 35.2433), 'Netherlands': (52.1326, 5.2913),
    'Taiwan': (23.6978, 120.9605), 'Dominican Republic': (18.7357, -70.1627),
    'Brazil': (-14.2350, -51.9253), 'Austria': (47.5162, 14.5501),
    'Kenya': (-0.0236, 37.9062), 'Philippines': (12.8797, 121.7740),
    'Hong Kong': (22.3193, 114.1694), 'Belgium': (50.5039, 4.4699),
    'Thailand': (15.8700, 100.9925), 'Jamaica': (18.1096, -77.2975),
    'South Korea': (35.9078, 127.7669), 'Switzerland': (46.8182, 8.2275),
    'Israel': (31.0461, 34.8516), 'Argentina': (-38.4161, -63.6167),
    'Greece': (39.0742, 21.8243), 'Nigeria': (9.0820, 8.6753),
    'Finland': (61.9241, 25.7482), 'Costa Rica': (9.7489, -83.7534),
    'Pakistan': (30.3753, 69.3451), 'Guatemala': (15.7835, -90.2308),
    'Hungary': (47.1625, 19.5033), 'South Africa': (-30.5595, 22.9375),
    'Ireland': (53.1424, -7.6921), 'Poland': (51.9194, 19.1451),
    'Tuvalu': (-7.1095, 177.6493), 'Fiji': (-16.5782, 179.4144),
    'Macedonia': (41.6086, 21.7453), 'Egypt': (26.8206, 30.8025),
    'Ukraine': (48.3794, 31.1656), 'New Zealand': (-40.9006, 174.8860),
    'Slovenia': (46.1512, 14.9955), 'Lithuania': (55.1694, 23.8813),
    'Latvia': (56.8796, 24.6032), 'Sweden': (60.1282, 18.6435),
    'Singapore': (1.3521, 103.8198), 'Bangladesh': (23.6850, 90.3563),
    'The Bahamas': (25.0343, -77.3963), 'Vatican City': (41.9029, 12.4534),
    'Czech Republic': (49.8175, 15.4730), 'Liberia': (6.4281, -9.4295),
    'Dominica': (15.4150, -61.3710), 'Norway': (60.4720, 8.4689),
    'Peru': (-9.1900, -75.0152), 'Uganda': (1.3733, 32.2903),
    'Portugal': (39.3999, -8.2245), 'Uruguay': (-32.5228, -55.7658),
    'Denmark': (56.2639, 9.5018), 'Zimbabwe': (-19.0154, 29.1549),
    'Syria': (34.8021, 38.9968), 'Indonesia': (-0.7893, 113.9213),
    'Nauru': (-0.5228, 166.9315), 'Croatia': (45.1000, 15.2000),
    'Barbados': (13.1939, -59.5432), 'Samoa': (-13.7590, -172.1046),
    'Chile': (-35.6751, -71.5430), 'Iceland': (64.9631, -19.0208),
    'Saudi Arabia': (23.8859, 45.0792), 'Suriname': (3.9193, -56.0278),
    'Tunisia': (33.8869, 9.5375), 'Luxembourg': (49.8153, 6.1296),
    'Cambodia': (12.5657, 104.9910), 'Russia': (61.5240, 105.3188),
    'East Timor (Timor-Leste)': (-8.8742, 125.7275), 'Ecuador': (-1.8312, -78.1834),
    'Nicaragua': (12.8654, -85.2072), 'Bulgaria': (42.7339, 25.4858),
    'Yemen': (15.5527, 48.5164), 'Niger': (17.6078, 8.0817),
    'Burkina Faso': (12.3641, -1.5330), 'Cabo Verde': (16.5388, -23.0418),
    'Vietnam': (14.0583, 108.2772), 'Slovakia': (48.6690, 19.6990),
    "Cote d'Ivoire": (7.5400, -5.5471), 'Kuwait': (29.3117, 47.4818),
    'Kosovo': (42.6026, 20.9030), 'Swaziland': (-26.5225, 31.4659),
    'Ghana': (7.9465, -1.0232), 'Bahrain': (26.0667, 50.5577),
    'Moldova': (47.4116, 28.3699), 'Myanmar (Burma)': (21.9162, 95.9560),
    'Uzbekistan': (41.3775, 64.5853), 'Panama': (8.5380, -80.7821),
    'Sri Lanka': (7.8731, 80.7718), 'Jordan': (30.5852, 36.2384),
    'Central African Republic': (6.6111, 20.9394), 'Eritrea': (15.1794, 39.7823),
    'Comoros': (-11.6455, 43.3333), 'Monaco': (43.7384, 7.4246),
    'Mozambique': (-18.6657, 35.5296), 'Estonia': (58.5953, 25.0136),
    'Belarus': (53.7098, 27.9534), 'Sudan': (12.8628, 30.2176),
    'Antigua and Barbuda': (17.0608, -61.7964), 'Honduras': (15.1999, -86.2419),
    'Bosnia and Herzegovina': (43.9159, 17.6791), 'Morocco': (31.7917, -7.0926),
    'Malta': (35.9375, 14.3754), 'Mongolia': (46.8625, 103.8467),
    'Senegal': (14.4974, -14.4524), 'South Sudan': (6.8770, 31.3070),
    'Somalia': (5.1521, 46.1996), 'Belize': (17.1899, -88.4976),
    'Lebanon': (33.8547, 35.8623), 'Venezuela': (6.4238, -66.5897),
    'Romania': (45.9432, 24.9668), 'Azerbaijan': (40.1431, 47.5769),
    'Guyana': (4.8604, -58.9302), 'Djibouti': (11.8251, 42.5903),
    'Seychelles': (-4.6796, 55.4920), 'Andorra': (42.5063, 1.5218),
    'Trinidad and Tobago': (10.6918, -61.2225), 'Qatar': (25.3548, 51.1839),
    'Montenegro': (42.7087, 19.3744)
}

# Combine both
all_coords = {**us_state_coords, **country_coords}

# Map to df
df['merchant_lat'] = df['Merchant State'].map(lambda x: all_coords.get(x, (np.nan, np.nan))[0])
df['merchant_lon'] = df['Merchant State'].map(lambda x: all_coords.get(x, (np.nan, np.nan))[1])

print(df[['Merchant State', 'merchant_lat', 'merchant_lon']].head(10))


  Merchant State  merchant_lat  merchant_lon
0            NaN           NaN           NaN
1             CA     36.116203   -119.681564
2             MS     32.741646    -89.678696
3            NaN           NaN           NaN
4            NaN           NaN           NaN
5             CA     36.116203   -119.681564
6             PA     40.590752    -77.209755
7            NaN           NaN           NaN
8             FL     27.766279    -81.686783
9          Italy     41.871900     12.567400


In [ ]:
FILL_LAT = 0  # replace this later
FILL_LON = 0  # replace this later

df['merchant_lat'] = df['merchant_lat'].fillna(FILL_LAT)
df['merchant_lon'] = df['merchant_lon'].fillna(FILL_LON)


print(df.head())

   User  Card  Year  Month  Day  Amount            Use Chip  \
0  1560     1  2013      7   18   20.24  Online Transaction   
1   488     0  2003      8   23   15.43   Swipe Transaction   
2  1634     2  2019      8   30    2.09    Chip Transaction   
3  1429     1  2018     11    2   52.47  Online Transaction   
4  1276     0  2019      9    9   37.68  Online Transaction   

         Merchant Name Merchant City Merchant State  ...  Errors?  Is Fraud?  \
0 -6458444334611773637        ONLINE            NaN  ...      NaN          0   
1  6621275923246397337   Los Angeles             CA  ...      NaN          0   
2 -6571010470072147219      Meridian             MS  ...      NaN          0   
3 -2088492411650162548        ONLINE            NaN  ...      NaN          0   
4  4241336128694185533        ONLINE            NaN  ...      NaN          0   

  Hour  merchant_freq  merchant_city_freq  merchant_state_enc  use_chip_enc  \
0   14           8715              168886                 186

In [ ]:
from math import radians, sin, cos, sqrt, atan2
import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# Sort by user and time
df = df.sort_values(['User', 'Year', 'Month', 'Day', 'Hour'])

# Previous lat/lon and fraud label per user
df['prev_lat'] = df.groupby('User')['merchant_lat'].shift(1)
df['prev_lon'] = df.groupby('User')['merchant_lon'].shift(1)
df['prev_fraud'] = df.groupby('User')['Is Fraud?'].shift(1)

# Distance only where previous OR current is fraud
df['fraud_dist'] = df.apply(
    lambda row: haversine(row['prev_lat'], row['prev_lon'], row['merchant_lat'], row['merchant_lon'])
    if pd.notna(row['prev_lat']) and (row['prev_fraud'] == 1 or row['Is Fraud?'] == 1)
    else 0,
    axis=1
)
df['non_fraud_dist'] = df.apply(
    lambda row: haversine(row['prev_lat'], row['prev_lon'], row['merchant_lat'], row['merchant_lon'])
    if pd.notna(row['prev_lat']) and (row['prev_fraud'] == 0 and row['Is Fraud?'] == 0)
    else 0,
    axis=1
)

df.drop(columns=['prev_lat', 'prev_lon', 'prev_fraud'], inplace=True)

print(df[['User', 'Is Fraud?', 'fraud_dist']].tail())

         User  Is Fraud?  fraud_dist
1080644  1999          0         0.0
27446    1999          0         0.0
321740   1999          0         0.0
732355   1999          0         0.0
510715   1999          0         0.0


In [ ]:
print(df[df['fraud_dist'] > 0]['fraud_dist'].describe())

count    16489.000000
mean      8989.835606
std       2818.032279
min         26.834542
25%       8147.988476
50%       9221.559416
75%      10310.343374
max      19182.665292
Name: fraud_dist, dtype: float64


In [ ]:
print(df[df['fraud_dist'] > 0].head(10))

         User  Card  Year  Month  Day  Amount            Use Chip  \
285118      0     3  2008     10   20  128.00  Online Transaction   
499801      0     0  2015     11   15  287.13  Online Transaction   
1269589     0     2  2015     11   22   15.20    Chip Transaction   
942489      0     3  2016      2   23   22.40   Swipe Transaction   
12354       0     2  2016      3    6  297.86  Online Transaction   
504045      0     1  2016      3   10   43.07    Chip Transaction   
516473      1     4  2010      3   22   77.96  Online Transaction   
1213407     1     4  2010      3   22  142.59   Swipe Transaction   
858081      1     4  2010      3   22   47.19  Online Transaction   
217538      1     2  2010      4    5  102.56   Swipe Transaction   

               Merchant Name   Merchant City Merchant State  ...  Hour  \
285118   1108327803852946055          ONLINE            NaN  ...    20   
499801  -8194607650924472520          ONLINE            NaN  ...    12   
1269589 -502349761

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, f1_score

# 2. Clean + encode
if df['Amount'].dtype == object:
    df['Amount'] = df['Amount'].str.replace('$', '', regex=False).astype(float)

if df['Is Fraud?'].dtype == object:
    df['Is Fraud?'] = df['Is Fraud?'].map({'Yes': 1, 'No': 0})

if 'Time' in df.columns:
    df['Hour'] = df['Time'].str.split(':').str[0].astype(int)
    df.drop(columns=['Time'], inplace=True)

if 'Merchant Name' in df.columns:
    df['merchant_freq'] = df['Merchant Name'].map(df['Merchant Name'].value_counts())

if 'Merchant City' in df.columns:
    df['merchant_city_freq'] = df['Merchant City'].map(df['Merchant City'].value_counts())

if 'Merchant State' in df.columns:
    df['merchant_state_enc'] = LabelEncoder().fit_transform(df['Merchant State'].astype(str))

if 'Use Chip' in df.columns:
    df['use_chip_enc'] = LabelEncoder().fit_transform(df['Use Chip'].astype(str))

if 'Errors?' in df.columns:
    df['has_error'] = df['Errors?'].fillna('').ne('').astype(int)

# 3. Drop raw text columns
cols_to_drop = [c for c in ['Use Chip', 'Merchant Name', 'Merchant City', 'Merchant State', 'Errors?', 'Zip'] if c in df.columns]
df_model = df.drop(columns=cols_to_drop)
print("Model-ready shape:", df_model.shape)
print(df_model.dtypes)

# 4. Split features/target
X = df_model.drop(columns=['Is Fraud?'])
y = df_model['Is Fraud?']

# 5. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

# 6. scale_pos_weight
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", scale_pos_weight)

# 7. Train XGBoost
model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    random_state=42
)
model.fit(X_train, y_train)

# 8. Evaluate
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, digits=4))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("PR-AUC:", average_precision_score(y_test, y_proba))


Model-ready shape: (1386900, 18)
User                    int64
Card                    int64
Year                    int64
Month                   int64
Day                     int64
Amount                float64
MCC                     int64
Is Fraud?               int64
Hour                    int64
merchant_freq           int64
merchant_city_freq      int64
merchant_state_enc      int64
use_chip_enc            int64
has_error               int64
merchant_lat          float64
merchant_lon          float64
fraud_dist            float64
non_fraud_dist        float64
dtype: object
Train: (1109520, 17) | Test: (277380, 17)
scale_pos_weight: 45.606737797193986

--- Classification Report ---
              precision    recall  f1-score   support

           0     0.9995    0.9951    0.9973    271429
           1     0.8136    0.9780    0.8883      5951

    accuracy                         0.9947    277380
   macro avg     0.9066    0.9865    0.9428    277380
weighted avg     0.9955    0.99

In [ ]:
import joblib

# Save model
joblib.dump(model, 'fraud_model.pkl')

# Load model later
model = joblib.load('fraud_model.pkl')

In [ ]:
from flask import Flask, request, jsonify, render_template_string
import joblib
import numpy as np
from pyngrok import ngrok

app = Flask(__name__)
model = joblib.load('fraud_model.pkl')

HTML = """
<!DOCTYPE html>
<html>
<head>
    <title>Fraud Detection</title>
    <style>
        body { font-family: Arial; max-width: 600px; margin: 50px auto; padding: 20px; }
        input { width: 100%; padding: 8px; margin: 8px 0; box-sizing: border-box; }
        button { background: #007bff; color: white; padding: 10px 20px; border: none; cursor: pointer; width: 100%; }
        #result { margin-top: 20px; padding: 15px; border-radius: 5px; text-align: center; font-size: 20px; }
        .fraud { background: #ffcccc; color: red; }
        .safe { background: #ccffcc; color: green; }
    </style>
</head>
<body>
    <h2>Credit Card Fraud Detection</h2>
    <input type="number" id="user" placeholder="User ID" />
    <input type="number" id="card" placeholder="Card Number" />
    <input type="number" id="year" placeholder="Year" />
    <input type="number" id="month" placeholder="Month" />
    <input type="number" id="day" placeholder="Day" />
    <input type="number" id="amount" placeholder="Transaction Amount" />
    <input type="number" id="mcc" placeholder="MCC Code" />
    <input type="number" id="hour" placeholder="Hour (0-23)" />
    <input type="number" id="merchant_freq" placeholder="Merchant Frequency" />
    <input type="number" id="merchant_city_freq" placeholder="Merchant City Frequency" />
    <input type="number" id="merchant_state_enc" placeholder="Merchant State Encoded" />
    <input type="number" id="use_chip_enc" placeholder="Use Chip Encoded" />
    <input type="number" id="has_error" placeholder="Has Error (0 or 1)" />
    <input type="number" id="merchant_lat" placeholder="Merchant Latitude" />
    <input type="number" id="merchant_lon" placeholder="Merchant Longitude" />
    <input type="number" id="fraud_dist" placeholder="Fraud Distance (km)" />
    <input type="number" id="non_fraud_dist" placeholder="Non Fraud Distance (km)" />
    <button onclick="predict()">Check Transaction</button>
    <div id="result"></div>

    <script>
        async function predict() {
            const data = {
                user: parseFloat(document.getElementById('user').value),
                card: parseFloat(document.getElementById('card').value),
                year: parseFloat(document.getElementById('year').value),
                month: parseFloat(document.getElementById('month').value),
                day: parseFloat(document.getElementById('day').value),
                amount: parseFloat(document.getElementById('amount').value),
                mcc: parseFloat(document.getElementById('mcc').value),
                hour: parseFloat(document.getElementById('hour').value),
                merchant_freq: parseFloat(document.getElementById('merchant_freq').value),
                merchant_city_freq: parseFloat(document.getElementById('merchant_city_freq').value),
                merchant_state_enc: parseFloat(document.getElementById('merchant_state_enc').value),
                use_chip_enc: parseFloat(document.getElementById('use_chip_enc').value),
                has_error: parseFloat(document.getElementById('has_error').value),
                merchant_lat: parseFloat(document.getElementById('merchant_lat').value),
                merchant_lon: parseFloat(document.getElementById('merchant_lon').value),
                fraud_dist: parseFloat(document.getElementById('fraud_dist').value),
                non_fraud_dist: parseFloat(document.getElementById('non_fraud_dist').value)
            };
            const response = await fetch('/predict', {
                method: 'POST',
                headers: { 'Content-Type': 'application/json' },
                body: JSON.stringify(data)
            });
            const result = await response.json();
            const div = document.getElementById('result');
            if (result.prediction === 1) {
                div.className = 'fraud';
                div.innerHTML = '⚠️ FRAUDULENT TRANSACTION (Probability: ' + result.fraud_probability + ')';
            } else {
                div.className = 'safe';
                div.innerHTML = '✅ LEGITIMATE TRANSACTION (Probability: ' + result.fraud_probability + ')';
            }
        }
    </script>
</body>
</html>
"""

@app.route('/')
def home():
    return render_template_string(HTML)

@app.route('/predict', methods=['POST'])
def predict():
    data = request.json
    features = np.array([[
        data['user'],
        data['card'],
        data['year'],
        data['month'],
        data['day'],
        data['amount'],
        data['mcc'],
        data['hour'],
        data['merchant_freq'],
        data['merchant_city_freq'],
        data['merchant_state_enc'],
        data['use_chip_enc'],
        data['has_error'],
        data['merchant_lat'],
        data['merchant_lon'],
        data['fraud_dist'],
        data['non_fraud_dist']
    ]])
    prediction = model.predict(features)[0]
    probability = model.predict_proba(features)[0][1]
    return jsonify({
        'prediction': int(prediction),
        'fraud_probability': round(float(probability), 4)
    })

ngrok.set_auth_token("3HXy3TUzidM7dzq0sZmm3dNqI1n_5Be3GfR22TPWw1CBfQLSL")
ngrok.kill()
public_url = ngrok.connect(5000)
print("Open this URL:", public_url)

app.run(port=5000)

Open this URL: NgrokTunnel: "https://dime-heap-down.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [06/Aug/2026 15:12:12] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [06/Aug/2026 15:12:26] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [06/Aug/2026 15:12:26] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [06/Aug/2026 15:12:26] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [06/Aug/2026 15:12:34] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [06/Aug/2026 15:17:27] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [06/Aug/2026 15:29:56] "GET / HTTP/1.1" 200 -


In [ ]:
print(X.columns.tolist())

['User', 'Card', 'Year', 'Month', 'Day', 'Amount', 'MCC', 'Hour', 'merchant_freq', 'merchant_city_freq', 'merchant_state_enc', 'use_chip_enc', 'has_error', 'merchant_lat', 'merchant_lon', 'fraud_dist', 'non_fraud_dist']
